# Maia Lite Chat 0.3.0 — Dataset Acquisition Pilot v8.1

**Executable Google Colab notebook** — integrates the dataset catalog CSV with the existing Corpus Builder v8.

This is a **metadata and acquisition pilot**, not a full corpus build. It:
1. Loads the catalog from Google Drive (or a bundled fallback catalog).
2. Validates source identifiers and distinguishes training candidates from evaluation datasets.
3. Queries Hugging Face metadata and available configurations.
4. Samples selected datasets **in streaming mode**, without downloading complete datasets.
5. Saves provenance, per-language quality estimates, token proxies, errors and a manifest.
6. **Never approves training rights automatically** and never exports unreviewed data to a training set.

**Run with CPU runtime first.** HF repositories and dataset schemas can change. Review errors rather than interpreting an empty sample as a dataset being empty.


**v8.2 correction:** Fixed the missing configuration handling for FineWeb2 and Wikipedia. The pilot now runs explicit PT/EN/ES jobs. The `production_ready` flag remains false until independent rights and quality reviews are completed.

In [ ]:
%pip -q install "datasets>=3.0,<5" "huggingface_hub>=0.26" "pandas>=2.0" "tqdm>=4.60" 

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
from collections import Counter, defaultdict
import csv, json, hashlib, re, time, os, random
import pandas as pd

try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    print("Outside Colab: using local path if available")

ROOT = Path("/content/drive/MyDrive/Colab_LLMs/Maia_Lite_0_3_0_Chat")
CATALOG_DIR = ROOT / "catalog"
CATALOG_PATH = CATALOG_DIR / "Maia_Lite_030_Dataset_Catalog.csv"
OUTPUT_DIR = ROOT / "corpus_v8" / "runs" / "acquisition_pilot_v8_1_001"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Bounded sampling — do not raise until you have inspected the first run.
MAX_ROWS_PER_DATASET = 150
MAX_SCAN_ROWS = 2500
MAX_CHARS_PER_DOC = 12000
MAX_SECONDS_PER_DATASET = 180
ALLOW_STREAMING = True
# Avoid accidentally querying very large sources on first run.
SELECTED_IDS = ["OpenAssistant/oasst1", "HuggingFaceFW/fineweb-2", "wikimedia/wikipedia"]
# Leave configuration overrides blank initially; update from discovered config report.
CONFIG_OVERRIDES = {}  # e.g. {"HuggingFaceFW/fineweb-2": "por_Latn"}
SPLIT_OVERRIDES = {}   # e.g. {"OpenAssistant/oasst1": "train"}
# Strictly a review ledger: changing this field never overrides the required evidence.
APPROVED_SOURCE_IDS = set()
print("Catalog:", CATALOG_PATH)
print("Output:", OUTPUT_DIR)

## 1. Catalog import and validation

The embedded fallback is the same catalog distributed with the acquisition plan. **Upload the CSV to the `catalog/` directory** to edit it independently. The notebook reads the Drive CSV preferentially.


In [ ]:
FALLBACK_CSV = '''priority,dataset_id,url,languages,proposed_domain,license_and_caveats,review_status,next_action
P0,HuggingFaceFW/fineweb-2,https://huggingface.co/datasets/HuggingFaceFW/fineweb-2,pt;es;multilingual,language,ODC-BY-1.0; Common Crawl terms,pending_review,streaming sample
P0,wikimedia/wikipedia,https://huggingface.co/datasets/wikimedia/wikipedia,pt;en;es,language;educational,CC-BY-SA/GFDL; page-specific,pending_review,streaming sample
P0,OpenAssistant/oasst1,https://huggingface.co/datasets/OpenAssistant/oasst1,pt-BR;en;es,dialogue,Apache-2.0,pending_review,small sample
P1,HuggingFaceFW/fineweb-edu,https://huggingface.co/datasets/HuggingFaceFW/fineweb-edu,en,educational,verify current terms and Common Crawl,pending_review,streaming sample
P1,allenai/peS2o,https://huggingface.co/datasets/allenai/peS2o,en,educational,ODC-BY; original sources review,pending_review,sample
P1,allenai/dolma,https://huggingface.co/datasets/allenai/dolma,en,language;educational,ODC-BY; original sources review,pending_review,shards only
P2,HuggingFaceH4/ultrachat_200k,https://huggingface.co/datasets/HuggingFaceH4/ultrachat_200k,en,dialogue_synthetic,MIT declared; generated data terms,pending_review,SFT candidate
P2,HuggingFaceTB/smol-smoltalk,https://huggingface.co/datasets/HuggingFaceTB/smol-smoltalk,en,dialogue_synthetic,Apache-2.0 declared,pending_review,SFT candidate
EVAL,openai/gsm8k,https://huggingface.co/datasets/openai/gsm8k,en,math_benchmark,MIT,reserved_for_eval,freeze test
EVAL,allenai/ai2_arc,https://huggingface.co/datasets/allenai/ai2_arc,en,reasoning_benchmark,CC-BY-SA-4.0,reserved_for_eval,freeze eval
'''
import io
if CATALOG_PATH.exists():
    df = pd.read_csv(CATALOG_PATH, dtype=str).fillna("")
    print("Loaded Drive catalog:", CATALOG_PATH)
else:
    CATALOG_DIR.mkdir(parents=True, exist_ok=True)
    CATALOG_PATH.write_text(FALLBACK_CSV, encoding="utf-8")
    df = pd.read_csv(io.StringIO(FALLBACK_CSV), dtype=str).fillna("")
    print("Created default catalog:", CATALOG_PATH)
required = {"priority","dataset_id","url","languages","proposed_domain","license_and_caveats","review_status","next_action"}
assert required.issubset(df.columns), f"Missing columns: {required-set(df.columns)}"
assert df["dataset_id"].is_unique, "Duplicate dataset IDs"
assert all(x.count("/") == 1 for x in df["dataset_id"]), "Invalid HF dataset ID"
assert not df["dataset_id"].str.strip().eq("").any(), "Empty dataset ID"
print(df[["priority","dataset_id","languages","review_status"]].to_string(index=False))
df.to_csv(OUTPUT_DIR/"catalog_snapshot.csv", index=False)

## 2. Live metadata and configuration discovery

License labels are **claims from the dataset metadata**, not independent legal certification. Configuration discovery can be expensive for enormous datasets; it is performed only for the selected pilot sources.


In [ ]:
from huggingface_hub import HfApi
from datasets import get_dataset_config_names, get_dataset_split_names, load_dataset
api = HfApi()
metadata = []
for dataset_id in df["dataset_id"]:
    row = df.loc[df.dataset_id == dataset_id].iloc[0].to_dict()
    record = {"dataset_id":dataset_id,"priority":row["priority"],
              "catalog_review_status":row["review_status"],
              "catalog_license_notes":row["license_and_caveats"],
              "metadata_status":"unknown","hub_license":None,
              "card_url":f"https://huggingface.co/datasets/{dataset_id}",
              "gated":None,"sha":None,"error":None}
    try:
        info = api.dataset_info(dataset_id)
        card = info.cardData or {}
        record.update(metadata_status="ok", hub_license=card.get("license"),
                      gated=info.gated, sha=info.sha, last_modified=str(info.last_modified))
    except Exception as e:
        record.update(metadata_status="error", error=f"{type(e).__name__}: {str(e)[:300]}")
    metadata.append(record)
    print(dataset_id, record["metadata_status"], "license:", record["hub_license"])
(OUTPUT_DIR/"hub_metadata.json").write_text(json.dumps(metadata,indent=2,ensure_ascii=False,default=str),encoding="utf-8")


In [ ]:
# Resolve an explicit config for each requested language; never assume "default".
LANGUAGE_JOBS = [
    {"dataset_id":"HuggingFaceFW/fineweb-2","language":"pt","config":"por_Latn"},
    {"dataset_id":"HuggingFaceFW/fineweb-2","language":"es","config":"spa_Latn"},
    {"dataset_id":"HuggingFaceFW/fineweb-2","language":"en","config":"eng_Latn"},
    {"dataset_id":"wikimedia/wikipedia","language":"pt","config":"20231101.pt"},
    {"dataset_id":"wikimedia/wikipedia","language":"en","config":"20231101.en"},
    {"dataset_id":"wikimedia/wikipedia","language":"es","config":"20231101.es"},
    {"dataset_id":"OpenAssistant/oasst1","language":"mixed","config":None},
]
# Edit LANGUAGE_JOBS to add/remove language-specific pilot jobs.
configs={}
for dataset_id in sorted({j["dataset_id"] for j in LANGUAGE_JOBS}):
    try:
        names=get_dataset_config_names(dataset_id,trust_remote_code=False)
        configs[dataset_id]={"status":"ok","count":len(names),
                             "requested_configs":{j["config"]:j["config"] in names for j in LANGUAGE_JOBS if j["dataset_id"]==dataset_id and j["config"]},
                             "first_20":names[:20]}
    except Exception as e:
        configs[dataset_id]={"status":"error","error":f"{type(e).__name__}: {str(e)[:400]}"}
    print(dataset_id,configs[dataset_id].get("requested_configs",configs[dataset_id].get("error")))
(OUTPUT_DIR/"config_discovery.json").write_text(json.dumps(configs,indent=2,ensure_ascii=False),encoding="utf-8")

## 3. Controlled streaming pilot

For a multilingual dataset, **never interpret the first N rows of an unspecified configuration as representative of PT/EN/ES**. The notebook uses explicit per-language jobs. Verify that each requested configuration exists in the discovery report.

This cell records raw samples **only for audit**. They are not approved training exports. Text is truncated for storage; text hashes are computed on the full available field.


In [ ]:
def extract_text(row):
    for field in ("text","content","body","document","raw_content"):
        if isinstance(row.get(field),str) and row[field].strip():
            return row[field],field
    messages=row.get("messages")
    if isinstance(messages,list):
        parts=[]
        for m in messages:
            if isinstance(m,dict):
                t=m.get("content") or m.get("text")
                if isinstance(t,str):parts.append(t)
        if parts:return "\n".join(parts),"messages"
    return None,None

def proxy_tokens(text):
    return max(1,round(len(text)/4))  # Proxy only; not the final tokenizer.

pilot=[]
audit_path=OUTPUT_DIR/"candidate_samples_audit.jsonl"
with audit_path.open("w",encoding="utf-8") as audit:
    for job in LANGUAGE_JOBS:
        dataset_id,lang,config=job["dataset_id"],job["language"],job["config"]
        job_id=f"{dataset_id}:{config or 'no_config'}"
        result={"job_id":job_id,"dataset_id":dataset_id,"language":lang,"config":config,
                "status":"not_run","rows_scanned":0,"rows_sampled":0,"proxy_tokens":0,"error":None}
        if dataset_id not in set(df.dataset_id):
            result["status"]="missing_from_catalog"
        elif df.loc[df.dataset_id==dataset_id,"priority"].iloc[0]=="EVAL":
            result["status"]="reserved_for_evaluation"
        elif not ALLOW_STREAMING:
            result["status"]="disabled"
        elif config and not configs.get(dataset_id,{}).get("requested_configs",{}).get(config,False):
            result["status"]="config_unavailable"
        elif configs.get(dataset_id,{}).get("status")!="ok":
            result["status"]="config_discovery_failed"
        else:
            try:
                # Name is passed explicitly when available; avoids implicit 'default'.
                kwargs={"streaming":True,"trust_remote_code":False}
                if config:kwargs["name"]=config
                if dataset_id in SPLIT_OVERRIDES:
                    split=SPLIT_OVERRIDES[dataset_id]
                else:
                    try:
                        splits=get_dataset_split_names(dataset_id,config_name=config,trust_remote_code=False)
                        split="train" if "train" in splits else (splits[0] if splits else None)
                    except Exception:
                        split="train"
                if not split:raise ValueError("No split found")
                result["split"]=split
                ds=load_dataset(dataset_id,split=split,**kwargs)
                started=time.monotonic()
                for idx,row in enumerate(ds):
                    if idx>=MAX_SCAN_ROWS or result["rows_sampled"]>=MAX_ROWS_PER_DATASET:break
                    if time.monotonic()-started>MAX_SECONDS_PER_DATASET:
                        result["status"]="time_budget_reached";break
                    result["rows_scanned"]+=1
                    content,field=extract_text(row)
                    if not content or len(content)<80:continue
                    result["rows_sampled"]+=1
                    result["proxy_tokens"]+=proxy_tokens(content)
                    audit.write(json.dumps({"job_id":job_id,"dataset_id":dataset_id,
                        "config":config,"split":split,"row_number":idx,"language_target":lang,
                        "language_verified":False if lang=="mixed" else None,
                        "text_field":field,"sha256":hashlib.sha256(content.encode()).hexdigest(),
                        "char_length":len(content),"proxy_tokens":proxy_tokens(content),
                        "text_excerpt":content[:MAX_CHARS_PER_DOC],
                        "rights_status":"pending_review","eligible_for_training":False},ensure_ascii=False)+"\n")
                if result["status"]=="not_run":result["status"]="sampled"
            except Exception as e:
                result["status"]="error"
                result["error"]=f"{type(e).__name__}: {str(e)[:500]}"
        pilot.append(result)
        print(job_id,result["status"],"rows",result["rows_sampled"],"proxy",result["proxy_tokens"])
(OUTPUT_DIR/"pilot_stats.json").write_text(json.dumps(pilot,indent=2,ensure_ascii=False),encoding="utf-8")

## 4. Rights gate, source coverage and manifest

A dataset with a `pending_review` status is **not** made eligible by a permissive-looking license field. Legal/source review and document-level restrictions must be completed separately. No candidate audit samples are fed into the v8 training export.


In [ ]:
TARGET_PARAMS=354_823_168
TARGET_TOKENS=TARGET_PARAMS*20*3
LANGUAGE_TARGETS={"pt":0.40,"en":0.35,"es":0.25}
DOMAINS={"language":0.30,"dialogue":0.20,"logic":0.25,"educational":0.15,"creative":0.10}
expected_cells={f"{lang}/{domain}" for lang in LANGUAGE_TARGETS for domain in DOMAINS}
# This pilot does not perform rights adjudication. All sampled documents remain ineligible.
approved_cells=set()
stats_by_id={p["dataset_id"]:p for p in pilot}
manifest={
    "version":"acquisition_pilot_v8_1",
    "generated_utc":datetime.now(timezone.utc).isoformat(),
    "root":str(ROOT),"catalog_path":str(CATALOG_PATH),
    "catalog_entries":len(df),
    "training_candidate_entries":int((df.priority!="EVAL").sum()),
    "evaluation_only_entries":int((df.priority=="EVAL").sum()),
    "metadata_ok":sum(m["metadata_status"]=="ok" for m in metadata),
    "pilot_datasets_requested":[j["dataset_id"]+":"+(j["config"] or "no_config") for j in LANGUAGE_JOBS],
    "pilot_datasets_sampled":sum(p["status"]=="sampled" for p in pilot),
    "pilot_datasets_needing_config":[p["job_id"] for p in pilot if p["status"] in ("config_unavailable","config_discovery_failed")],
    "pilot_errors":{p["job_id"]:p["error"] for p in pilot if p["error"]},
    "pilot_proxy_tokens":sum(p["proxy_tokens"] for p in pilot),
    "token_proxy_warning":"Character-count proxy, not tokenizer count; pilot sample not representative of total dataset.",
    "target_processed_tokens":TARGET_TOKENS,
    "target_tokens_by_language":{k:round(TARGET_TOKENS*v) for k,v in LANGUAGE_TARGETS.items()},
    "approved_training_tokens":0,
    "missing_approved_cells":sorted(expected_cells-approved_cells),
    "rights_approved":False,
    "production_ready":False,
    "blockers":[
        "No source/document rights decisions performed by this notebook",
        "Representative per-language samples and total unique-token yield not yet measured",
        "Source-family and near-duplicate audits pending",
        "Frozen contamination benchmark not supplied",
        "Final tokenizer not trained or certified"
    ],
    "outputs":["catalog_snapshot.csv","hub_metadata.json","config_discovery.json",
               "candidate_samples_audit.jsonl","pilot_stats.json","manifest_acquisition_v8_1.json"]
}
manifest_path=OUTPUT_DIR/"manifest_acquisition_v8_1.json"
manifest_path.write_text(json.dumps(manifest,indent=2,ensure_ascii=False),encoding="utf-8")
print("Manifest:",manifest_path)
print("Production ready:",manifest["production_ready"])
print("Datasets needing explicit configuration:",manifest["pilot_datasets_needing_config"])
print("Pilot errors:",manifest["pilot_errors"])
print("Observed pilot proxy tokens:",manifest["pilot_proxy_tokens"])

## 5. Next steps after the first run

1. Inspect `config_discovery.json`. Inspect the explicit PT/EN/ES jobs in `LANGUAGE_JOBS` and `config_discovery.json`; adjust unavailable configurations as needed.
2. Inspect `hub_metadata.json` and the **original dataset cards/source policies**. Record legal review decisions outside the notebook; do not infer approval from the catalog's license label.
3. Compare sampled text quality and source diversity. The initial sample is **not a representative volume estimate**.
4. Freeze evaluation benchmarks and run source-family and approximate-deduplication audits.
5. Only after review, import eligible sources into the Corpus Builder v8. Keep candidate samples and training exports in separate directories.

**Important:** This notebook does not download 21.29B tokens. It establishes verifiable acquisition jobs and diagnostics before scaling.
